# Extract custom CNN or VGG16 features
Inputs are resized to 384 × 384 RGB and rescaled by 1/255, as in the historical cache-generation procedure. Custom CNN extraction requires a trained Dense(256) model. A Dense32 development model is rejected. The NPZ includes its label order.

In [ ]:
from pathlib import Path
import sys, zipfile, subprocess
PROJECT_ROOT = Path('/content/DMAE_Colab_Repository')
if not PROJECT_ROOT.exists():
    local = Path.cwd()
    if (local / 'src' / 'amd_oct').exists():
        PROJECT_ROOT = local
    else:
        from google.colab import files
        print('Upload DMAE_Colab_Repository.zip (code package, not the image database).')
        uploaded = files.upload()
        package = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
        if package is None:
            raise ValueError('Select the code ZIP package.')
        destination = Path('/content').resolve()
        with zipfile.ZipFile(package) as archive:
            for member in archive.infolist():
                resolved = (destination / member.filename).resolve()
                if not resolved.is_relative_to(destination):
                    raise ValueError('Unsafe archive path.')
            archive.extractall(destination)
if not (PROJECT_ROOT / 'src' / 'amd_oct').is_dir():
    raise FileNotFoundError('Set PROJECT_ROOT to the extracted code-package directory.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Code root:', PROJECT_ROOT)


In [ ]:
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/AMD_OCT') if USE_GOOGLE_DRIVE else Path('/content/AMD_OCT')
DATASET_DIR = BASE_DIR / 'data'  # Contains Normal, iORA, cORA, iRORA, cRORA folders.
CACHE_DIR = BASE_DIR / 'features'
MODEL_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
for directory in (CACHE_DIR, MODEL_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import importlib.util
if importlib.util.find_spec('tensorflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-neural.txt')])


In [ ]:
from amd_oct.features import extract_features
EXTRACTOR = 'custom_cnn'  # 'custom_cnn' or 'vgg16'
MODEL_PATH = MODEL_DIR / 'new_custom_cnn_run' / 'custom_cnn_256.keras'
OUTPUT_PATH = CACHE_DIR / f'{EXTRACTOR}_features.npz'
cache_path = extract_features(DATASET_DIR, OUTPUT_PATH, EXTRACTOR,
    model_path=MODEL_PATH if EXTRACTOR == 'custom_cnn' else None)
print('Saved:', cache_path)
